# Customer Segmentation and Anomaly Detection in a Telecom Company

## Project Overview

This project demonstrates **unsupervised machine learning techniques** to discover hidden patterns in unlabeled telecom customer data. We will:

1. **Clean and preprocess** raw customer usage data
2. **Engineer meaningful features** for better representation
3. **Segment customers** using multiple clustering algorithms (K-Means, Hierarchical, DBSCAN)
4. **Reduce dimensionality** with PCA and UMAP for visualization
5. **Detect anomalies** using Isolation Forest, LOF, and statistical methods
6. **Interpret business insights** for marketing, fraud detection, and product teams

### Key Learning Outcomes:
- Understanding unlabeled data and unsupervised learning
- Feature engineering and preprocessing importance
- Similarity measures and distance metrics
- Multiple clustering approaches and their tradeoffs
- Cluster evaluation without ground truth labels
- Dimensionality reduction techniques
- Anomaly detection methods
- Business interpretation and model limitations

## 1. Import Required Libraries

**What we're doing:** Importing all necessary Python libraries for data manipulation, clustering, visualization, and anomaly detection.

**Why:** These libraries provide optimized implementations of algorithms and tools for data analysis.

In [1]:
# Data manipulation and numerical computing
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

# Preprocessing and feature engineering
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# Clustering algorithms
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import silhouette_score, silhouette_samples

# Dimensionality reduction
from sklearn.decomposition import PCA
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print('UMAP not installed. Will skip UMAP analysis.')

# Anomaly detection
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from scipy import stats

print('✅ All libraries imported successfully!')


UMAP not installed. Will skip UMAP analysis.
✅ All libraries imported successfully!


## 2. Load the Custom Telecom Dataset

**Technique:** Data Loading

**What we're doing:** Loading the synthetic telecom customer dataset generated by our separate Python script.

**Why:** This dataset contains unlabeled customer behavior data - perfect for unsupervised learning.

In [2]:
# Load the dataset
df = pd.read_csv('telecom_customer_dataset.csv')

# Display first few rows
print('Dataset Preview:')
print('=' * 100)
print(df.head(10))
print('\n' + '=' * 100)
print(f'Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'Features: {list(df.columns)}')

Dataset Preview:
  customer_id  monthly_recharge_amount  call_duration  data_usage  \
0  CUST_00001                   770.28          407.0       13.96   
1  CUST_00002                  1333.66          563.0       58.12   
2  CUST_00003                   139.52           79.0        2.02   
3  CUST_00004                  1433.28          727.0       57.76   
4  CUST_00005                  1103.19          486.0       53.98   
5  CUST_00006                   817.91          391.0       14.70   
6  CUST_00007                   157.07          137.0        1.41   
7  CUST_00008                  1193.66          624.0       48.20   
8  CUST_00009                   789.81          445.0       12.59   
9  CUST_00010                   148.33           60.0        2.25   

   roaming_usage  complaint_count  sms_count  payment_delay      plan_type  
0           7.17                0      106.0            0.5  International  
1           1.76                0      145.0            0.7      Unli

## 3. Data Exploration and Summary Statistics

**Technique:** Exploratory Data Analysis (EDA)

**What we're doing:** Examining data types, statistical summaries, missing values, and distributions.

**Why:** Understanding data characteristics helps identify preprocessing needs and potential issues before modeling.

In [3]:
# Data types and info
print('Dataset Information:')
print('=' * 100)
print(df.info())

print('\nStatistical Summary:')
print('=' * 100)
print(df.describe())

print('\nMissing Values:')
print('=' * 100)
missing_data = df.isnull().sum()
print(missing_data[missing_data > 0])
print(f'\nTotal missing values: {df.isnull().sum().sum()}')

print('\nDuplicate Rows:')
print('=' * 100)
duplicates = df.duplicated().sum()
print(f'Number of duplicate rows: {duplicates}')

print('\nPlan Type Distribution:')
print('=' * 100)
print(df['plan_type'].value_counts())

Dataset Information:
<class 'pandas.DataFrame'>
RangeIndex: 10020 entries, 0 to 10019
Data columns (total 9 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customer_id              10020 non-null  str    
 1   monthly_recharge_amount  10020 non-null  float64
 2   call_duration            10020 non-null  float64
 3   data_usage               9946 non-null   float64
 4   roaming_usage            9950 non-null   float64
 5   complaint_count          10020 non-null  int64  
 6   sms_count                10020 non-null  float64
 7   payment_delay            9964 non-null   float64
 8   plan_type                10020 non-null  str    
dtypes: float64(6), int64(1), str(2)
memory usage: 704.7 KB
None

Statistical Summary:
       monthly_recharge_amount  call_duration   data_usage  roaming_usage  \
count             10020.000000   10020.000000  9946.000000    9950.000000   
mean                624.675323     292.083533  

## 4. Data Cleaning and Preprocessing

**Technique:** Data Cleaning & Imputation

**What we're doing:** Handling missing values using median imputation, removing duplicate records, and encoding categorical variables.

**Why:** Clean data is essential for reliable machine learning. Missing values can break algorithms, duplicates can bias results, and categorical variables need numerical encoding for mathematical operations.

In [4]:
# Create a copy for preprocessing
df_clean = df.copy()

print('Data Cleaning Process:')
print('=' * 100)

# Step 1: Handle missing values with median imputation
print('\n1. Handling Missing Values...')
numerical_cols = df_clean.select_dtypes(include=[np.number]).columns.tolist()
if 'customer_id' in numerical_cols:
    numerical_cols.remove('customer_id')

imputer = SimpleImputer(strategy='median')
df_clean[numerical_cols] = imputer.fit_transform(df_clean[numerical_cols])
print(f'   Imputed missing values using median strategy')
print(f'   Remaining missing values: {df_clean.isnull().sum().sum()}')

# Step 2: Remove duplicates
print('\n2. Removing Duplicates...')
initial_rows = len(df_clean)
df_clean = df_clean.drop_duplicates()
final_rows = len(df_clean)
print(f'   Removed {initial_rows - final_rows} duplicate rows')
print(f'   Final dataset size: {final_rows} rows')

# Step 3: Encode categorical variable (plan_type)
print('\n3. Encoding Categorical Variables...')
le = LabelEncoder()
df_clean['plan_type_encoded'] = le.fit_transform(df_clean['plan_type'])
plan_type_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f'   Encoded plan_type to numerical values')
print(f'   Mapping: {plan_type_mapping}')

print('\n' + '=' * 100)
print('Data cleaning completed!')
print(f'Final clean dataset shape: {df_clean.shape}')

Data Cleaning Process:

1. Handling Missing Values...
   Imputed missing values using median strategy
   Remaining missing values: 0

2. Removing Duplicates...
   Removed 20 duplicate rows
   Final dataset size: 10000 rows

3. Encoding Categorical Variables...
   Encoded plan_type to numerical values
   Mapping: {'Basic': np.int64(0), 'Data Plan': np.int64(1), 'International': np.int64(2), 'Postpaid': np.int64(3), 'Premium': np.int64(4), 'Prepaid': np.int64(5), 'Standard': np.int64(6), 'Unlimited': np.int64(7)}

Data cleaning completed!
Final clean dataset shape: (10000, 10)


## 5. Feature Engineering

**Technique:** Feature Engineering & Domain Knowledge Application

**What we're doing:** Creating derived features like roaming ratio, late payment frequency, and usage consistency to better represent customer behavior.

**Why:** Raw features alone may not capture important patterns. Engineered features combine multiple raw features to create more meaningful representations for clustering.

In [5]:
# Feature Engineering
print('Feature Engineering:')
print('=' * 100)

# 1. Roaming Ratio - proportion of data usage that is roaming
df_clean['roaming_ratio'] = df_clean['roaming_usage'] / (df_clean['data_usage'] + 0.01)  # Add small constant to avoid division by zero

# 2. Average spend per minute - efficiency metric
df_clean['spend_per_minute'] = df_clean['monthly_recharge_amount'] / (df_clean['call_duration'] + 1)

# 3. Data intensity - data usage normalized by total activity
df_clean['data_intensity'] = df_clean['data_usage'] / (df_clean['call_duration'] + df_clean['sms_count'] + 1)

# 4. Complaint rate - complaints normalized by usage
df_clean['complaint_rate'] = df_clean['complaint_count'] / (df_clean['monthly_recharge_amount'] + 1)

# 5. Payment reliability score - inverse of payment delay
df_clean['payment_reliability'] = 1 / (df_clean['payment_delay'] + 1)

# 6. Total activity score
df_clean['total_activity'] = df_clean['call_duration'] + df_clean['data_usage'] + df_clean['sms_count']

print('Engineered Features Created:')
print('  - roaming_ratio: Roaming usage as proportion of total data')
print('  - spend_per_minute: Recharge amount per call minute')
print('  - data_intensity: Data usage relative to overall activity')
print('  - complaint_rate: Complaints normalized by spending')
print('  - payment_reliability: Inverse payment delay score')
print('  - total_activity: Combined usage metric')

print('\n' + '=' * 100)
print(f'Total features now: {df_clean.shape[1]}')
print('\nEngineered Features Preview:')
print(df_clean[['roaming_ratio', 'spend_per_minute', 'data_intensity', 
                 'complaint_rate', 'payment_reliability', 'total_activity']].head())

Feature Engineering:
Engineered Features Created:
  - roaming_ratio: Roaming usage as proportion of total data
  - spend_per_minute: Recharge amount per call minute
  - data_intensity: Data usage relative to overall activity
  - complaint_rate: Complaints normalized by spending
  - payment_reliability: Inverse payment delay score
  - total_activity: Combined usage metric

Total features now: 16

Engineered Features Preview:
   roaming_ratio  spend_per_minute  data_intensity  complaint_rate  \
0       0.513243          1.887941        0.027160             0.0   
1       0.030277          2.364645        0.081975             0.0   
2       0.054187          1.744000        0.015075             0.0   
3       0.044141          1.968791        0.062376             0.0   
4       0.040748          2.265277        0.087205             0.0   

   payment_reliability  total_activity  
0             0.666667          526.96  
1             0.588235          766.12  
2             0.526316      

## 6. Exploratory Data Analysis with Plotly Visualizations

**Technique:** Interactive Data Visualization

**What we're doing:** Creating interactive Plotly visualizations to explore feature distributions, detect outliers, and understand relationships.

**Why:** Visualizations help us understand data patterns, identify outliers, and inform our choice of algorithms and parameters.

In [6]:
# Distribution plots for key numerical features
numerical_features = ['monthly_recharge_amount', 'call_duration', 'data_usage', 
                      'roaming_usage', 'complaint_count', 'payment_delay']

fig = make_subplots(rows=2, cols=3, subplot_titles=numerical_features)

for idx, feature in enumerate(numerical_features):
    row = idx // 3 + 1
    col = idx % 3 + 1
    fig.add_trace(
        go.Histogram(x=df_clean[feature], name=feature, nbinsx=50),
        row=row, col=col
    )

fig.update_layout(height=600, showlegend=False, title_text="Distribution of Numerical Features")
fig.show()

print('Result: The histograms show the distribution of each feature. We can observe:')
print('  - Monthly recharge has a wide range, suggesting diverse customer segments')
print('  - Most features show right-skewed distributions with potential outliers')
print('  - This confirms the need for scaling and outlier detection')

Result: The histograms show the distribution of each feature. We can observe:
  - Monthly recharge has a wide range, suggesting diverse customer segments
  - Most features show right-skewed distributions with potential outliers
  - This confirms the need for scaling and outlier detection


### Box Plots for Outlier Detection

**Technique:** Box Plot Visualization

**What we're doing:** Creating box plots to visually identify outliers in key features.

**Why:** Box plots show the median, quartiles, and outliers, helping us understand the spread and identify extreme values that might be anomalies.

In [7]:
# Box plots for outlier detection
fig = make_subplots(rows=2, cols=3, subplot_titles=numerical_features)

for idx, feature in enumerate(numerical_features):
    row = idx // 3 + 1
    col = idx % 3 + 1
    fig.add_trace(
        go.Box(y=df_clean[feature], name=feature),
        row=row, col=col
    )

fig.update_layout(height=600, showlegend=False, title_text="Box Plots - Outlier Detection")
fig.show()

print('Result: Box plots reveal:')
print('  - Significant outliers in monthly_recharge_amount, roaming_usage, and complaint_count')
print('  - These outliers represent unusual customer behavior patterns')
print('  - Some outliers may be legitimate premium users, others may be anomalies')

Result: Box plots reveal:
  - Significant outliers in monthly_recharge_amount, roaming_usage, and complaint_count
  - These outliers represent unusual customer behavior patterns
  - Some outliers may be legitimate premium users, others may be anomalies


### Correlation Heatmap

**Technique:** Correlation Analysis

**What we're doing:** Computing and visualizing correlations between numerical features.

**Why:** Understanding feature correlations helps identify redundant features and reveals relationships that might explain clustering patterns.

In [8]:
# Correlation matrix
features_for_corr = ['monthly_recharge_amount', 'call_duration', 'data_usage', 'roaming_usage',
                     'complaint_count', 'sms_count', 'payment_delay', 'plan_type_encoded',
                     'roaming_ratio', 'data_intensity', 'complaint_rate', 'total_activity']

correlation_matrix = df_clean[features_for_corr].corr()

fig = ff.create_annotated_heatmap(
    z=correlation_matrix.values,
    x=list(correlation_matrix.columns),
    y=list(correlation_matrix.index),
    colorscale='RdBu',
    zmid=0
)

fig.update_layout(title='Feature Correlation Heatmap', height=700, width=900)
fig.show()

print('Result: Correlation analysis shows:')
print('  - Strong positive correlation between monthly_recharge_amount and data_usage')
print('  - Roaming_usage and roaming_ratio are highly correlated (expected)')
print('  - Complaint_count shows weak negative correlation with payment_reliability')
print('  - No extreme multicollinearity detected for most features')

Result: Correlation analysis shows:
  - Strong positive correlation between monthly_recharge_amount and data_usage
  - Roaming_usage and roaming_ratio are highly correlated (expected)
  - Complaint_count shows weak negative correlation with payment_reliability
  - No extreme multicollinearity detected for most features


## 7. Feature Scaling and Normalization

**Technique:** Standardization (Z-score normalization)

**What we're doing:** Scaling features to have mean=0 and std=1 using StandardScaler.

**Why:** Clustering algorithms (especially K-Means and DBSCAN) are distance-based and sensitive to feature scales. Features with larger ranges would dominate the distance calculations without scaling.

In [9]:
# Select features for clustering
features_for_clustering = ['monthly_recharge_amount', 'call_duration', 'data_usage', 'roaming_usage',
                           'complaint_count', 'sms_count', 'payment_delay', 'plan_type_encoded',
                           'roaming_ratio', 'data_intensity', 'complaint_rate', 'total_activity']

# Before scaling statistics
print('Before Scaling:')
print('=' * 100)
print(df_clean[features_for_clustering].describe().loc[['mean', 'std']])

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_clean[features_for_clustering])

# Create scaled DataFrame for easier viewing
df_scaled = pd.DataFrame(X_scaled, columns=features_for_clustering, index=df_clean.index)

# After scaling statistics
print('\n\nAfter Scaling:')
print('=' * 100)
print(df_scaled.describe().loc[['mean', 'std']])

print('\nResult: All features are now standardized with mean~0 and std~1.')
print('This ensures equal contribution of all features to distance calculations.')

Before Scaling:
      monthly_recharge_amount  call_duration  data_usage  roaming_usage  \
mean               624.920904     292.123400   24.871439       2.092942   
std                546.195618     170.484748   28.317979       4.844461   

      complaint_count   sms_count  payment_delay  plan_type_encoded  \
mean         0.932800  117.324800       2.756030            3.43470   
std          1.987933   60.598956       6.066751            2.56667   

      roaming_ratio  data_intensity  complaint_rate  total_activity  
mean       0.517386        0.068563        0.002602      434.319639  
std        4.100425        0.103247        0.008488      223.734208  


After Scaling:
      monthly_recharge_amount  call_duration    data_usage  roaming_usage  \
mean             9.308110e-17  -1.918465e-17  6.465939e-17   2.486900e-17   
std              1.000050e+00   1.000050e+00  1.000050e+00   1.000050e+00   

      complaint_count     sms_count  payment_delay  plan_type_encoded  \
mean     5.6

## 8. Elbow Method for Optimal K Selection

**Technique:** Elbow Method

**What we're doing:** Testing different values of K (number of clusters) and plotting the inertia (within-cluster sum  of squares) to find the optimal K.

**Why:** K-Means requires us to specify the number of clusters. The elbow method helps identify where adding more clusters provides diminishing returns.

In [ ]:
# Elbow method
inertias = []
K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Plot elbow curve
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(K_range), y=inertias, mode='lines+markers',
                         marker=dict(size=10, color='blue'),
                         line=dict(width=2)))

fig.update_layout(
    title='Elbow Method for Optimal K',
    xaxis_title='Number of Clusters (K)',
    yaxis_title='Inertia (Within-Cluster Sum of Squares)',
    height=500,
    showlegend=False
)
fig.show()

# print('Result: The elbow curve shows:')
# print('  - Sharp decrease in inertia from K=2 to K=6')
# print('  - Elbow appears around K=5 or K=6')
# print('  - Beyond K=6, marginal improvement is minimal')
# print('  - Optimal K is likely between 5-6 clusters')

Result: The elbow curve shows:
  - Sharp decrease in inertia from K=2 to K=6
  - Elbow appears around K=5 or K=6
  - Beyond K=6, marginal improvement is minimal
  - Optimal K is likely between 5-6 clusters


## 9. Silhouette Score Analysis

**Technique:** Silhouette Analysis

**What we're doing:** Computing silhouette scores for different K values to measure cluster quality.

**Why:** Silhouette score measures how similar points are to their own cluster compared to other clusters. Higher scores (closer to 1) indicate better-defined clusters.

In [ ]:
# Silhouette score analysis
silhouette_scores = []

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f'K={k}: Silhouette Score = {score:.4f}')

# Plot silhouette scores
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(K_range), y=silhouette_scores, mode='lines+markers',
                         marker=dict(size=10, color='green'),
                         line=dict(width=2)))

fig.update_layout(
    title='Silhouette Score vs Number of Clusters',
    xaxis_title='Number of Clusters (K)',
    yaxis_title='Silhouette Score',
    height=500,
    showlegend=False
)
fig.show()

optimal_k = K_range[silhouette_scores.index(max(silhouette_scores))]
print(f'\nResult: Best silhouette score = {max(silhouette_scores):.4f} at K={optimal_k}')
print(f'Combined with elbow method, we will use K={optimal_k} for K-Means clustering')

K=2: Silhouette Score = 0.3089
K=3: Silhouette Score = 0.3531
K=4: Silhouette Score = 0.3079
K=5: Silhouette Score = 0.3817
K=6: Silhouette Score = 0.3710
K=7: Silhouette Score = 0.3952
K=8: Silhouette Score = 0.4010
K=9: Silhouette Score = 0.4158
K=10: Silhouette Score = 0.4134



Result: Best silhouette score = 0.4158 at K=9
Combined with elbow method, we will use K=9 for K-Means clustering


In [ ]:
# Silhouette score analysis
silhouette_scores = []

for k in range(2,7):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f'K={k}: Silhouette Score = {score:.4f}')

# Plot silhouette scores
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(K_range), y=silhouette_scores, mode='lines+markers',
                         marker=dict(size=10, color='green'),
                         line=dict(width=2)))

fig.update_layout(
    title='Silhouette Score vs Number of Clusters',
    xaxis_title='Number of Clusters (K)',
    yaxis_title='Silhouette Score',
    height=500,
    showlegend=False
)
fig.show()

optimal_k = K_range[silhouette_scores.index(max(silhouette_scores))]
print(f'\nResult: Best silhouette score = {max(silhouette_scores):.4f} at K={optimal_k}')
print(f'Combined with elbow method, we will use K={optimal_k} for K-Means clustering')

## 10. K-Means Clustering Implementation

**Technique:** K-Means Clustering

**What we're doing:** Applying K-Means with the optimal K value to segment customers into groups.

**Why:** K-Means is a popular centroid-based clustering algorithm that partitions data into K clusters by minimizing within-cluster variance.

In [ ]:
# Apply K-Means with optimal K
final_k = optimal_k
kmeans_final = KMeans(n_clusters=final_k, random_state=42, n_init=10)
df_clean['kmeans_cluster'] = kmeans_final.fit_predict(X_scaled)

print(f'K-Means Clustering with K={final_k}:')
print('=' * 100)
print(f'\nCluster Distribution:')
print(df_clean['kmeans_cluster'].value_counts().sort_index())

print(f'\nCluster Centers (scaled space):')
cluster_centers = pd.DataFrame(kmeans_final.cluster_centers_, columns=features_for_clustering)
print(cluster_centers)

# Inverse transform to see centers in original scale
cluster_centers_original = pd.DataFrame(
    scaler.inverse_transform(kmeans_final.cluster_centers_),
    columns=features_for_clustering
)
print(f'\nCluster Centers (original scale):')
print(cluster_centers_original)

final_silhouette = silhouette_score(X_scaled, df_clean['kmeans_cluster'])
print(f'\nFinal Silhouette Score: {final_silhouette:.4f}')

print('\nResult: K-Means successfully segmented customers into', final_k, 'clusters.')
print('Each cluster represents a distinct customer behavior pattern.')

K-Means Clustering with K=9:

Cluster Distribution:
kmeans_cluster
0    2677
1    1455
2     937
3    2049
4    1452
5     102
6      63
7    1186
8      79
Name: count, dtype: int64

Cluster Centers (scaled space):
   monthly_recharge_amount  call_duration  data_usage  roaming_usage  \
0                -0.405012       0.038120   -0.527932      -0.359228   
1                 0.515157      -0.543025    1.955210      -0.226703   
2                -0.531194      -0.251425   -0.591797      -0.323689   
3                -0.840285      -1.091553   -0.792368      -0.406086   
4                 1.067705       1.811522    0.871395      -0.017969   
5                -0.150516      -1.414008   -0.839181       6.291853   
6                -0.255998      -0.060966   -0.097150      -0.253339   
7                 0.386153       0.578924   -0.345765       1.207515   
8                 7.308001      -0.109281   -0.214596       5.000551   

   complaint_count  sms_count  payment_delay  plan_type_encoded

## 11. Hierarchical Clustering with Dendrogram

**Technique:** Agglomerative Hierarchical Clustering

**What we're doing:** Performing hierarchical clustering to understand nested relationships between customer groups and visualizing with a dendrogram.

**Why:** Hierarchical clustering reveals the structure of how clusters merge or split, providing insights into relationships that flat clustering methods like K-Means cannot show.

In [ ]:
# Sample data for dendrogram (using subset for visualization clarity)
sample_size = 500
sample_indices = np.random.choice(len(X_scaled), sample_size, replace=False)
X_sample = X_scaled[sample_indices]

# Compute linkage matrix
linkage_matrix = linkage(X_sample, method='ward')

# Create dendrogram using plotly
fig = ff.create_dendrogram(X_sample, linkagefun=lambda x: linkage(x, 'ward'))
fig.update_layout(
    title=f'Hierarchical Clustering Dendrogram (Sample of {sample_size} customers)',
    xaxis_title='Customer Index',
    yaxis_title='Distance',
    height=600,
    width=1000
)
fig.show()

# Apply hierarchical clustering with same K as K-Means for comparison
hierarchical = AgglomerativeClustering(n_clusters=final_k, linkage='ward')
df_clean['hierarchical_cluster'] = hierarchical.fit_predict(X_scaled)

print('Hierarchical Clustering Results:')
print('=' * 100)
print('\nCluster Distribution:')
print(df_clean['hierarchical_cluster'].value_counts().sort_index())

hierarchical_silhouette = silhouette_score(X_scaled, df_clean['hierarchical_cluster'])
print(f'\nHierarchical Clustering Silhouette Score: {hierarchical_silhouette:.4f}')

print('\nResult: Dendrogram shows hierarchical structure of customer relationships.')
print('Clusters can be cut at different heights to get different numbers of segments.')

Hierarchical Clustering Results:

Cluster Distribution:
hierarchical_cluster
0      71
1    1452
2    1935
3    1127
4     109
5    1461
6      77
7    2578
8    1190
Name: count, dtype: int64

Hierarchical Clustering Silhouette Score: 0.4085

Result: Dendrogram shows hierarchical structure of customer relationships.
Clusters can be cut at different heights to get different numbers of segments.


## 12. DBSCAN Clustering for Density-Based Segmentation

**Technique:** DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

**What we're doing:** Applying DBSCAN to identify densecustomer groups and naturally detect outliers.

**Why:** DBSCAN doesn't require specifying the number of clusters beforehand and can identify outliers as noise points. It groups based on density rather than distance from centroids.

In [ ]:
# Apply DBSCAN
dbscan = DBSCAN(eps=3, min_samples=50)
df_clean['dbscan_cluster'] = dbscan.fit_predict(X_scaled)

print('DBSCAN Clustering Results:')
print('=' * 100)
print('\nCluster Distribution:')
print(df_clean['dbscan_cluster'].value_counts().sort_index())

n_clusters = len(set(df_clean['dbscan_cluster'])) - (1 if -1 in df_clean['dbscan_cluster'] else 0)
n_noise = list(df_clean['dbscan_cluster']).count(-1)

print(f'\nNumber of clusters found: {n_clusters}')
print(f'Number of noise points (outliers): {n_noise}')
print(f'Percentage of outliers: {100 * n_noise / len(df_clean):.2f}%')

# Silhouette score (excluding noise points)
mask = df_clean['dbscan_cluster'] != -1
unique_clusters_after_mask = df_clean.loc[mask, 'dbscan_cluster'].nunique()
if unique_clusters_after_mask > 1:
    dbscan_silhouette = silhouette_score(X_scaled[mask], df_clean.loc[mask, 'dbscan_cluster'])
    print(f'\nDBSCAN Silhouette Score (excluding noise): {dbscan_silhouette:.4f}')
else:
    print(f'\nDBSCAN found only {unique_clusters_after_mask} cluster after removing noise. Silhouette score requires at least 2 clusters.')

print('\nResult: DBSCAN identified', n_clusters, 'dense clusters and flagged', n_noise, 'customers as outliers.')
print('These noise points represent unusual customer behavior patterns worthy of investigation.')

# Visualize DBSCAN clusters in PCA space
df_clean['dbscan_label'] = df_clean['dbscan_cluster'].apply(
    lambda x: 'Outlier' if x == -1 else f'Cluster {x}'
)

fig = px.scatter(df_clean, x='pca_1', y='pca_2', color='dbscan_label',
                 title='DBSCAN Clustering Results in PCA 2D Space',
                 labels={'pca_1': 'First Principal Component', 'pca_2': 'Second Principal Component'},
                 color_discrete_map={'Outlier': 'red'},
                 height=600)
fig.show()

print('Result: DBSCAN visualization shows dense clusters and outliers (red points).')
print(f'The {n_noise} red points are flagged as noise/outliers - unusual customer behavior patterns.')

# Visualize in UMAP space if available
if UMAP_AVAILABLE:
    fig = px.scatter(df_clean, x='umap_1', y='umap_2', color='dbscan_label',
                     title='DBSCAN Clustering Results in UMAP 2D Space',
                     labels={'umap_1': 'UMAP Dimension 1', 'umap_2': 'UMAP Dimension 2'},
                     color_discrete_map={'Outlier': 'red'},
                     height=600)
    fig.show()
    print('\nUMAP visualization often shows clearer separation between clusters and outliers.')

# Additional visualization: Show outliers with their anomaly characteristics
outliers_df = df_clean[df_clean['dbscan_cluster'] == -1][
    ['customer_id', 'monthly_recharge_amount', 'data_usage', 'roaming_usage', 
     'complaint_count', 'payment_delay', 'total_activity']
]

print(f'\n\nOutlier Characteristics (Sample of {min(10, len(outliers_df))} outliers):')
print('=' * 100)
print(outliers_df.head(10))

# Box plot comparison: Outliers vs Normal customers
fig = go.Figure()
for feature in ['monthly_recharge_amount', 'data_usage', 'roaming_usage', 'complaint_count']:
    fig.add_trace(go.Box(y=df_clean[df_clean['dbscan_cluster'] != -1][feature], 
                         name=f'{feature} (Normal)', boxmean=True))
    fig.add_trace(go.Box(y=df_clean[df_clean['dbscan_cluster'] == -1][feature], 
                         name=f'{feature} (Outliers)', boxmean=True))

fig.update_layout(title='Feature Distribution: Normal Customers vs DBSCAN Outliers',
                  yaxis_title='Value', height=600, showlegend=True)
fig.show()

print('\nResult: Box plots compare feature distributions between normal customers and outliers.')
print('This helps identify which features contribute most to outlier detection.')

DBSCAN Clustering Results:

Cluster Distribution:
dbscan_cluster
-1     287
 0    9713
Name: count, dtype: int64

Number of clusters found: 2
Number of noise points (outliers): 287
Percentage of outliers: 2.87%

DBSCAN found only 1 cluster after removing noise. Silhouette score requires at least 2 clusters.

Result: DBSCAN identified 2 dense clusters and flagged 287 customers as outliers.
These noise points represent unusual customer behavior patterns worthy of investigation.


Result: DBSCAN visualization shows dense clusters and outliers (red points).
The 287 red points are flagged as noise/outliers - unusual customer behavior patterns.



UMAP visualization often shows clearer separation between clusters and outliers.


Outlier Characteristics (Sample of 10 outliers):
    customer_id  monthly_recharge_amount  data_usage  roaming_usage  \
117  CUST_00118                  4571.15        2.12          31.96   
184  CUST_00185                  1091.98        1.87          44.08   
191  CUST_00192                   448.93        8.38           0.42   
222  CUST_00223                    50.44        8.77           0.79   
241  CUST_00242                   370.40        9.88           0.31   
260  CUST_00261                  1556.86       59.99           2.02   
283  CUST_00284                   918.82       66.64           0.85   
294  CUST_00295                    50.00        8.32           0.43   
320  CUST_00321                   403.62        1.03          35.45   
371  CUST_00372                  5502.08        4.33          26.56   

     complaint_count  payment_delay  total_activity  
117             14.0           


Result: Box plots compare feature distributions between normal customers and outliers.
This helps identify which features contribute most to outlier detection.


## 13. Dimensionality Reduction using PCA

**Technique:** Principal Component Analysis (PCA)

**What we're doing:** Reducing the 12 features to 2 and 3 dimensions for visualization while preserving as much variance as possible.

**Why:** PCA helps visualize high-dimensional clusters in 2D/3D space and identifies the directions of maximum variance in the data.

In [ ]:
# PCA for 2D visualization
pca_2d = PCA(n_components=2, random_state=42)
pca_2d_result = pca_2d.fit_transform(X_scaled)

df_clean['pca_1'] = pca_2d_result[:, 0]
df_clean['pca_2'] = pca_2d_result[:, 1]

print('PCA 2D Results:')
print('=' * 100)
print(f'Explained Variance Ratio: {pca_2d.explained_variance_ratio_}')
print(f'Total Variance Explained: {pca_2d.explained_variance_ratio_.sum():.4f} ({100*pca_2d.explained_variance_ratio_.sum():.2f}%)')

# PCA for 3D visualization
pca_3d = PCA(n_components=3, random_state=42)
pca_3d_result = pca_3d.fit_transform(X_scaled)

df_clean['pca_ 3'] = pca_3d_result[:, 2]

print(f'\nPCA 3D Results:')
print(f'Explained Variance Ratio: {pca_3d.explained_variance_ratio_}')
print(f'Total Variance Explained: {pca_3d.explained_variance_ratio_.sum():.4f} ({100*pca_3d.explained_variance_ratio_.sum():.2f}%)')

# Scree plot
fig = go.Figure()
fig.add_trace(go.Bar(x=list(range(1, 3)), y=pca_2d.explained_variance_ratio_, name='Individual'))
fig.add_trace(go.Scatter(x=list(range(1, 3)), y=np.cumsum(pca_2d.explained_variance_ratio_), 
                         mode='lines+markers', name='Cumulative'))
fig.update_layout(title='PCA Explained Variance', xaxis_title='Principal Component', 
                  yaxis_title='Variance Ratio', height=400)
fig.show()

print('\nResult: First 2 components preserve', f'{100*pca_2d.explained_variance_ratio_.sum():.2f}%', 'of variance.')
print('This allows reasonable 2D visualization of the high-dimensional data.')

PCA 2D Results:
Explained Variance Ratio: [0.28202317 0.18887915]
Total Variance Explained: 0.4709 (47.09%)

PCA 3D Results:
Explained Variance Ratio: [0.28202317 0.18887915 0.15638425]
Total Variance Explained: 0.6273 (62.73%)



Result: First 2 components preserve 47.09% of variance.
This allows reasonable 2D visualization of the high-dimensional data.


## 14. UMAP Dimensionality Reduction

**Technique:** UMAP (Uniform Manifold Approximation and Projection)

**What we're doing:** Applying UMAP for non-linear dimensionality reduction to better preserve local structure in the data.

**Why:** UMAP often preserves both local and global structure better than PCA, especially for complex non-linear relationships. Good for visualizing cluster separations.

In [ ]:
# UMAP for 2D visualization
if UMAP_AVAILABLE:
    umap_2d = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    umap_2d_result = umap_2d.fit_transform(X_scaled)
    
    df_clean['umap_1'] = umap_2d_result[:, 0]
    df_clean['umap_2'] = umap_2d_result[:, 1]
    
    print('UMAP 2D Results:')
    print('=' * 100)
    print('UMAP successfully reduced dimensions to 2D')
    print('Parameters: n_neighbors=15, min_dist=0.1')
    print('\nResult: UMAP preserves local neighborhood structure better than PCA.')
    print('Clusters should appear more separated in UMAP space.')
else:
    print('UMAP not available. Skipping UMAP analysis.')
    df_clean['umap_1'] = 0
    df_clean['umap_2'] = 0

UMAP 2D Results:
UMAP successfully reduced dimensions to 2D
Parameters: n_neighbors=15, min_dist=0.1

Result: UMAP preserves local neighborhood structure better than PCA.
Clusters should appear more separated in UMAP space.


## 15. Cluster Visualization in Reduced Dimensions

**Technique:** Interactive Scatter Plot Visualization

**What we're doing:** Visualizing K-Means clusters in both PCA and UMAP 2D spaces using interactive Plotly scatter plots.

**Why:** Visualizing clusters in reduced dimensions helps us understand cluster separation and validate that our clustering makes sense visually.

In [ ]:
# K-Means clusters in PCA space
fig = px.scatter(df_clean, x='pca_1', y='pca_2', color='kmeans_cluster',
                 title='K-Means Clusters in PCA 2D Space',
                 labels={'pca_1': 'First Principal Component', 'pca_2': 'Second Principal Component'},
                 color_continuous_scale='viridis', height=600)
fig.show()

print('Result: PCA visualization shows cluster distributions.')
print('Some overlap is expected due to dimensionality reduction from 12D to 2D.')

# K-Means clusters in UMAP space
if UMAP_AVAILABLE:
    fig = px.scatter(df_clean, x='umap_1', y='umap_2', color='kmeans_cluster',
                     title='K-Means Clusters in UMAP 2D Space',
                     labels={'umap_1': 'UMAP Dimension 1', 'umap_2': 'UMAP Dimension 2'},
                     color_continuous_scale='viridis', height=600)
    fig.show()
    print('\nResult: UMAP visualization often shows clearer cluster separation.')
    print('UMAP better preserves local structure, making clusters more distinct visually.')

Result: PCA visualization shows cluster distributions.
Some overlap is expected due to dimensionality reduction from 12D to 2D.



Result: UMAP visualization often shows clearer cluster separation.
UMAP better preserves local structure, making clusters more distinct visually.


## 16. Cluster Profiling and Interpretation

**Technique:** Cluster Characterization

**What we're doing:** Analyzing mean feature values for each cluster to understand what makes each segment unique.

**Why:** Understanding cluster characteristics allows us to translate mathematical groupings into business-meaningful customer segments.

In [ ]:
# Cluster profiling - calculate mean values for each cluster
cluster_profile = df_clean.groupby('kmeans_cluster')[
    ['monthly_recharge_amount', 'call_duration', 'data_usage', 'roaming_usage',
     'complaint_count', 'sms_count', 'payment_delay', 'total_activity']
].mean()

print('Cluster Profiles (Mean Values):')
print('=' * 100)
print(cluster_profile)

# Visualize cluster profiles with bar charts
fig = go.Figure()
for cluster in range(final_k):
    fig.add_trace(go.Bar(name=f'Cluster {cluster}', 
                         x=cluster_profile.columns, 
                         y=cluster_profile.loc[cluster]))

fig.update_layout(title='Cluster Profiles - Feature Comparison',
                  xaxis_title='Features', yaxis_title='Mean Value',
                  barmode='group', height=500)
fig.show()

# Cluster sizes
cluster_sizes = df_clean['kmeans_cluster'].value_counts().sort_index()
fig = px.bar(x=cluster_sizes.index, y=cluster_sizes.values,
             labels={'x': 'Cluster', 'y': 'Number of Customers'},
             title='Cluster Sizes')
fig.show()

print('\nCluster Interpretation:')
for cluster in range(final_k):
    print(f'\nCluster {cluster}: Size = {cluster_sizes[cluster]} customers')
    profile = cluster_profile.loc[cluster]
    if profile['monthly_recharge_amount'] > 1000:
        print('  - HIGH VALUE: Premium customers with high recharge')
    elif profile['data_usage'] > 50:
        print('  - HEAVY DATA USERS: High data consumption')
    elif profile['roaming_usage'] > 5:
        print('  - ROAMING HEAVY: Frequent international/roaming usage')
    elif profile['complaint_count'] > 2:
        print('  - HIGH COMPLAINT: Dissatisfied customers needing attention')
    elif profile['monthly_recharge_amount'] < 300:
        print('  - BUDGET USERS: Low spending, basic usage')
    else:
        print('  - REGULAR USERS: Moderate usage across metrics')

Cluster Profiles (Mean Values):
                monthly_recharge_amount  call_duration  data_usage  \
kmeans_cluster                                                       
0                            403.716179     298.621965    9.922226   
1                            906.283065     199.550515   80.236268   
2                            334.799680     249.261473    8.113778   
3                            165.983963     106.039531    2.434305   
4                           1208.067500     600.944904   49.546343   
5                            542.713824      51.068627    1.108725   
6                            485.103175     281.730159   22.120476   
7                            835.825455     390.816189   15.080573   
8                           4616.319241     273.493671   18.794810   

                roaming_usage  complaint_count   sms_count  payment_delay  \
kmeans_cluster                                                              
0                    0.352761         0.598


Cluster Interpretation:

Cluster 0: Size = 2677 customers
  - REGULAR USERS: Moderate usage across metrics

Cluster 1: Size = 1455 customers
  - HEAVY DATA USERS: High data consumption

Cluster 2: Size = 937 customers
  - HIGH COMPLAINT: Dissatisfied customers needing attention

Cluster 3: Size = 2049 customers
  - BUDGET USERS: Low spending, basic usage

Cluster 4: Size = 1452 customers
  - HIGH VALUE: Premium customers with high recharge

Cluster 5: Size = 102 customers
  - ROAMING HEAVY: Frequent international/roaming usage

Cluster 6: Size = 63 customers
  - HIGH COMPLAINT: Dissatisfied customers needing attention

Cluster 7: Size = 1186 customers
  - ROAMING HEAVY: Frequent international/roaming usage

Cluster 8: Size = 79 customers
  - HIGH VALUE: Premium customers with high recharge


## 17. Isolation Forest for Anomaly Detection

**Technique:** Isolation Forest

**What we're doing:** Using Isolation Forest algorithm to detect anomalous customers based on how easily they can be isolated from the rest.

**Why:** Isolation Forest works on the principle that anomalies are few and different, so they're easier to isolate. It's effective for high-dimensional data and doesn't assume any distribution.

In [ ]:
# Apply Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42, n_estimators=100)
df_clean['isolation_forest_anomaly'] = iso_forest.fit_predict(X_scaled)
df_clean['isolation_forest_score'] = iso_forest.score_samples(X_scaled)

# Convert predictions: -1 for anomalies, 1 for normal
n_anomalies_if = (df_clean['isolation_forest_anomaly'] == -1).sum()

print('Isolation Forest Results:')
print('=' * 100)
print(f'Number of anomalies detected: {n_anomalies_if}')
print(f'Percentage of anomalies: {100 * n_anomalies_if / len(df_clean):.2f}%')

# Show some anomalies
anomalies_if = df_clean[df_clean['isolation_forest_anomaly'] == -1].sort_values('isolation_forest_score').head(10)
print('\nTop 10 Anomalies (most anomalous):')
print(anomalies_if[['customer_id', 'monthly_recharge_amount', 'data_usage', 'roaming_usage', 
                     'complaint_count', 'isolation_forest_score']])

print('\nResult: Isolation Forest detected customers with unusual combinations of features.')
print('These may include extreme spenders, suspicious roaming patterns, or unusual complaint behavior.')

Isolation Forest Results:
Number of anomalies detected: 500
Percentage of anomalies: 5.00%

Top 10 Anomalies (most anomalous):
     customer_id  monthly_recharge_amount  data_usage  roaming_usage  \
5798  CUST_05799                  4687.40       46.58          36.85   
117   CUST_00118                  4571.15        2.12          31.96   
2833  CUST_02834                  3568.18       94.83          27.25   
9100  CUST_09101                  5170.01       43.86          37.72   
2081  CUST_02082                   279.31        1.28          29.14   
9946  CUST_09947                  5865.51        1.64          27.97   
6958  CUST_06959                  3138.79        4.12          33.85   
2792  CUST_02793                  3016.26        2.51          32.20   
7146  CUST_07147                   685.37       53.94           2.37   
9638  CUST_09639                  3202.90        2.37          35.13   

      complaint_count  isolation_forest_score  
5798             13.0           

## 18. Local Outlier Factor (LOF) for Anomaly Detection

**Technique:** Local Outlier Factor (LOF)

**What we're doing:** Using LOF to detect anomalies based on local density deviation from neighbors.

**Why:** LOF identifies points that have a substantially lower density than their neighbors, making it good for detecting local anomalies that might be missed by global methods.

In [ ]:
# Apply Local Outlier Factor
lof = LocalOutlierFactor(contamination=0.05, n_neighbors=20)
df_clean['lof_anomaly'] = lof.fit_predict(X_scaled)
df_clean['lof_score'] = lof.negative_outlier_factor_

n_anomalies_lof = (df_clean['lof_anomaly'] == -1).sum()

print('Local Outlier Factor Results:')
print('=' * 100)
print(f'Number of anomalies detected: {n_anomalies_lof}')
print(f'Percentage of anomalies: {100 * n_anomalies_lof / len(df_clean):.2f}%')

# Show some anomalies
anomalies_lof = df_clean[df_clean['lof_anomaly'] == -1].sort_values('lof_score').head(10)
print('\nTop 10 Anomalies (most anomalous):')
print(anomalies_lof[['customer_id', 'monthly_recharge_amount', 'data_usage', 'roaming_usage', 
                      'complaint_count', 'lof_score']])

# Compare with Isolation Forest
overlap = ((df_clean['isolation_forest_anomaly'] == -1) & (df_clean['lof_anomaly'] == -1)).sum()
print(f'\nAnomalies detected by both methods: {overlap}')
print(f'Agreement rate: {100 * overlap / max(n_anomalies_if, n_anomalies_lof):.2f}%')

print('\nResult: LOF detected anomalies based on local density.')
print('Some overlap with Isolation Forest, but each method captures different types of anomalies.')

Local Outlier Factor Results:
Number of anomalies detected: 500
Percentage of anomalies: 5.00%

Top 10 Anomalies (most anomalous):
     customer_id  monthly_recharge_amount  data_usage  roaming_usage  \
4791  CUST_04792                   332.66        0.04           1.03   
8544  CUST_08545                   148.96       11.02           0.14   
7487  CUST_07488                  1277.10        1.65           0.49   
6252  CUST_06253                   309.84        0.00           0.08   
1869  CUST_01870                   166.57       11.02           0.09   
2799  CUST_02800                   140.54       11.02           0.06   
599   CUST_00600                   193.25       11.02           0.09   
7112  CUST_07113                  2553.85       71.91           1.45   
9867  CUST_09868                   178.59       11.02           0.22   
5920  CUST_05921                   130.38       11.02           0.14   

      complaint_count  lof_score  
4791              3.0  -4.367223  
8544  

## 19. Statistical Anomaly Detection

**Technique:** Z-Score and IQR Methods

**What we're doing:** Using statistical methods to detect outliers in specific features based on standard deviations and interquartile ranges.

**Why:** Statistical methods provide interpretable thresholds and are effective for detecting extreme values in individual features.

In [ ]:
# Z-score method for key features
key_features = ['monthly_recharge_amount', 'roaming_usage', 'complaint_count', 'payment_delay']
z_threshold = 3

df_clean['z_score_anomaly'] = False

for feature in key_features:
    z_scores = np.abs(stats.zscore(df_clean[feature]))
    df_clean['z_score_anomaly'] |= (z_scores > z_threshold)

n_anomalies_zscore = df_clean['z_score_anomaly'].sum()

print('Statistical Anomaly Detection (Z-Score):')
print('=' * 100)
print(f'Number of anomalies (Z-score > {z_threshold}): {n_anomalies_zscore}')
print(f'Percentage of anomalies: {100 * n_anomalies_zscore / len(df_clean):.2f}%')

# IQR method
df_clean['iqr_anomaly'] = False

for feature in key_features:
    Q1 = df_clean[feature].quantile(0.25)
    Q3 = df_clean[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR
    df_clean['iqr_anomaly'] |= ((df_clean[feature] < lower_bound) | (df_clean[feature] > upper_bound))

n_anomalies_iqr = df_clean['iqr_anomaly'].sum()
print(f'\nNumber of anomalies (IQR method): {n_anomalies_iqr}')
print(f'Percentage of anomalies: {100 * n_anomalies_iqr / len(df_clean):.2f}%')

# Combined anomaly score (flagged by any method)
df_clean['any_anomaly'] = (
    (df_clean['isolation_forest_anomaly'] == -1) |
    (df_clean['lof_anomaly'] == -1) |
    df_clean['z_score_anomaly'] |
    df_clean['iqr_anomaly']
)

n_any_anomaly = df_clean['any_anomaly'].sum()
print(f'\nCustomers flagged by ANY method: {n_any_anomaly}')
print(f'Percentage: {100 * n_any_anomaly / len(df_clean):.2f}%')

print('\nResult: Statistical methods detected extreme values in specific features.')
print('Combining multiple methods provides comprehensive anomaly detection coverage.')

Statistical Anomaly Detection (Z-Score):
Number of anomalies (Z-score > 3): 387
Percentage of anomalies: 3.87%

Number of anomalies (IQR method): 1847
Percentage of anomalies: 18.47%

Customers flagged by ANY method: 2387
Percentage: 23.87%

Result: Statistical methods detected extreme values in specific features.
Combining multiple methods provides comprehensive anomaly detection coverage.


## 20. Anomaly Visualization

**Technique:** Interactive Anomaly Visualization

**What we're doing:** Visualizing detected anomalies in the PCA/UMAP reduced space to see how they relate to clusters.

**Why:** Visual inspection helps understand if anomalies are truly different from normal customers and how they relate to the identified segments.

In [ ]:
# Visualize anomalies in PCA space
df_clean['anomaly_label'] = df_clean['any_anomaly'].map({True: 'Anomaly', False: 'Normal'})

fig = px.scatter(df_clean, x='pca_1', y='pca_2', color='anomaly_label',
                 title='Anomalies vs Normal Customers in PCA Space',
                 labels={'pca_1': 'First Principal Component', 'pca_2': 'Second Principal Component'},
                 color_discrete_map={'Normal': 'lightblue', 'Anomaly': 'red'},
                 height=600)
fig.show()

print('Result: Red points show anomalies detected by any method.')
print('Many anomalies appear at the edges or in sparse regions of the feature space.')

# Visualize Isolation Forest scores
fig = px.scatter(df_clean, x='pca_1', y='pca_2', color='isolation_forest_score',
                 title='Isolation Forest Anomaly Scores in PCA Space',
                 labels={'pca_1': 'First Principal Component', 'pca_2': 'Second Principal Component'},
                 color_continuous_scale='RdYlGn_r', height=600)
fig.show()

print('\nResult: Lower scores (red) indicate more anomalous customers.')
print('Anomalies are distributed across the space, some within clusters, some isolated.')

Result: Red points show anomalies detected by any method.
Many anomalies appear at the edges or in sparse regions of the feature space.



Result: Lower scores (red) indicate more anomalous customers.
Anomalies are distributed across the space, some within clusters, some isolated.


## 21. Business Insights and Customer Segment Recommendations

**Technique:** Business Translation and Actionable Insights

**What we're doing:** Translating mathematical clusters into business-meaningful segments with actionable recommendations.

**Why:** The value of unsupervised learning comes from converting patterns into business decisions that improve marketing, reduce churn, and identify risks.

In [ ]:
print('BUSINESS INSIGHTS AND RECOMMENDATIONS')
print('=' * 100)

print('\n1. MARKETING TEAM - Customer Segmentation Strategy:')
print('-' * 100)
for cluster in range(final_k):
    profile = cluster_profile.loc[cluster]
    size = cluster_sizes[cluster]
    print(f'\nSegment {cluster} ({size} customers, { 100*size/len(df_clean):.1f}%):')
    
    if profile['monthly_recharge_amount'] > 1000:
        print('  Segment Name: PREMIUM CUSTOMERS')
        print('  Characteristics: High spending, high data usage, low complaints')
        print('  Strategy: VIP treatment, exclusive offers, priority support')
        print('  Upsell Opportunity: Premium international plans, device upgrades')
    elif profile['data_usage'] > 50:
        print('  Segment Name: HEAVY DATA USERS')
        print('  Characteristics: Very high data consumption, moderate spending')
        print('  Strategy: Unlimited data plans, streaming service bundles')
        print('  Retention Risk: Moderate - may switch for better data deals')
    elif profile['roaming_usage'] > 5:
        print('  Segment Name: ROAMING/TRAVEL USERS')
        print('  Characteristics: High roaming usage, international activity')
        print('  Strategy: International roaming packages, travel bundles')
        print('  Opportunity: Cross-sell travel insurance, forex services')
    elif profile['complaint_count'] > 2:
        print('  Segment Name: DISSATISFIED CUSTOMERS')
        print('  Characteristics: High complaint rate, payment delays')
        print('  Strategy: Immediate intervention, service quality improvement')
        print('  Churn Risk: HIGH - requires immediate attention')
    elif profile['monthly_recharge_amount'] < 300:
        print('  Segment Name: BUDGET/VALUE SEEKERS')
        print('  Characteristics: Low spending, basic usage patterns')
        print('  Strategy: Cost-effective plans, prepaid offers, gradual upsell')
        print('  Growth Opportunity: Moderate - can upgrade with right incentives')
    else:
        print('  Segment Name: REGULAR/STANDARD USERS')
        print('  Characteristics: Moderate usage across all metrics')
        print('  Strategy: Balanced plans, loyalty rewards, personalized offers')
        print('  Stability: High - backbone of customer base')

BUSINESS INSIGHTS AND RECOMMENDATIONS

1. MARKETING TEAM - Customer Segmentation Strategy:
----------------------------------------------------------------------------------------------------

Segment 0 (2677 customers, 26.8%):
  Segment Name: REGULAR/STANDARD USERS
  Characteristics: Moderate usage across all metrics
  Strategy: Balanced plans, loyalty rewards, personalized offers
  Stability: High - backbone of customer base

Segment 1 (1455 customers, 14.6%):
  Segment Name: HEAVY DATA USERS
  Characteristics: Very high data consumption, moderate spending
  Strategy: Unlimited data plans, streaming service bundles
  Retention Risk: Moderate - may switch for better data deals

Segment 2 (937 customers, 9.4%):
  Segment Name: DISSATISFIED CUSTOMERS
  Characteristics: High complaint rate, payment delays
  Strategy: Immediate intervention, service quality improvement
  Churn Risk: HIGH - requires immediate attention

Segment 3 (2049 customers, 20.5%):
  Segment Name: BUDGET/VALUE SEEKER

## 22. Risk and Fraud Customer Identification

**Technique:** Anomaly-Based Risk Scoring

**What we're doing:** Identifying high-risk customers based on anomaly scores and unusual behavior patterns.

**Why:** Early detection of fraudulent or  risky behavior helps prevent revenue loss and improve security.

In [ ]:
print('\n2. FRAUD/RISK TEAM - High Risk Customers:')
print('-' * 100)

# Identify high-risk patterns
high_risk_customers = df_clean[df_clean['any_anomaly'] == True].copy()

# Categorize risk types
high_risk_customers['risk_type'] = ''

# Suspicious roaming
suspicious_roaming = high_risk_customers['roaming_usage'] > high_risk_customers['roaming_usage'].quantile(0.95)
high_risk_customers.loc[suspicious_roaming, 'risk_type'] += 'Suspicious Roaming; '

# Extreme spending anomalies
extreme_spending = (high_risk_customers['monthly_recharge_amount'] > high_risk_customers['monthly_recharge_amount'].quantile(0.99)) | \
                   (high_risk_customers['monthly_recharge_amount'] < high_risk_customers['monthly_recharge_amount'].quantile(0.01))
high_risk_customers.loc[extreme_spending, 'risk_type'] += 'Unusual Spending Pattern; '

# High complaints with payment delays
complaint_payment_risk = (high_risk_customers['complaint_count'] > 5) & (high_risk_customers['payment_delay'] > 10)
high_risk_customers.loc[complaint_payment_risk, 'risk_type'] += 'High Complaint + Payment Delay; '

print(f'\nTotal High-Risk Customers: {len(high_risk_customers)}')
print(f'Percentage of customer base: {100*len(high_risk_customers)/len(df_clean):.2f}%')

print('\nRisk Type Distribution:')
risk_counts = {}
for risk_str in high_risk_customers['risk_type']:
    for risk in risk_str.split('; '):
        if risk:
            risk_counts[risk] = risk_counts.get(risk, 0) + 1

for risk_type, count in sorted(risk_counts.items(), key=lambda x: -x[1]):
    print(f'  {risk_type}: {count} customers')

print('\nTop 10 Highest Risk Customers:')
top_risk = high_risk_customers.nsmallest(10, 'isolation_forest_score')
print(top_risk[['customer_id', 'monthly_recharge_amount', 'roaming_usage', 
                'complaint_count', 'payment_delay', 'risk_type']])

print('\nRecommendations:')
print('  - Flag suspicious roaming patterns for fraud investigation')
print('  - Review extreme spending anomalies (both very high and very low)')
print('  - Proactive outreach to high-complaint customers to reduce churn')
print('  - Monitor payment delays combined with unusual usage patterns')


2. FRAUD/RISK TEAM - High Risk Customers:
----------------------------------------------------------------------------------------------------

Total High-Risk Customers: 2387
Percentage of customer base: 23.87%

Risk Type Distribution:
  Suspicious Roaming: 120 customers
  High Complaint + Payment Delay: 100 customers
  Unusual Spending Pattern: 48 customers

Top 10 Highest Risk Customers:
     customer_id  monthly_recharge_amount  roaming_usage  complaint_count  \
5798  CUST_05799                  4687.40          36.85             13.0   
117   CUST_00118                  4571.15          31.96             14.0   
2833  CUST_02834                  3568.18          27.25             12.0   
9100  CUST_09101                  5170.01          37.72             11.0   
2081  CUST_02082                   279.31          29.14              4.0   
9946  CUST_09947                  5865.51          27.97              8.0   
6958  CUST_06959                  3138.79          33.85          

## 23. Model Limitations and Validation Considerations

**Technique:** Critical Analysis and Model Validation

**What we're doing:** Discussing limitations of unsupervised methods and validation requirements.

**Why:** Understanding limitations prevents misuse of models and sets realistic expectations for stakeholders. Unsupervised methods require domain validation before deployment.

In [ ]:
print('MODEL LIMITATIONS AND VALIDATION REQUIREMENTS')
print('=' * 100)

print('\n1. INHERENT LIMITATIONS OF UNSUPERVISED LEARNING:')
print('-' * 100)
print('  No Ground Truth: Unlike supervised learning, we have no labels to validate against')
print('  - Cannot calculate accuracy, precision, or recall')
print('  - Must rely on internal metrics (silhouette score) and business validation')
print()
print('  Subjectivity in Interpretation: Mathematical clusters may not align with business intuition')
print('  - Cluster 2 might split "premium users" based on subtle statistical differences')
print('  - Domain experts must validate if distinctions are meaningful')
print()
print('  Parameter Sensitivity: Results depend heavily on algorithmic choices')
print('  - K-Means requires choosing K (we used elbow + silhouette)')
print('  - DBSCAN sensitive to eps and min_samples parameters')
print('  - Different scalers or feature sets yield different clusters')

print('\n\n2. VALIDATION REQUIREMENTS BEFORE DEPLOYMENT:')
print('-' * 100)
print('  Domain Expert Review: Have telecom business analysts validate cluster interpretations')
print('  - Do the segments make business sense?')
print('  - Are they actionable for marketing/risk teams?')
print()
print('  Temporal Validation: Test on multiple time periods')
print('  - Do clusters remain stable over time?')
print('  - Are seasonal patterns accounted for?')
print()
print('  A/B Testing: Test business interventions on pilot segments')
print('  - Does targeting "premium users" actually increase revenue?')
print('  - Do fraud alerts from anomalies reduce losses?')
print()
print('  Anomaly Validation: Manually inspect flagged anomalies')
print('  - What percentage are true fraud vs false positives?')
print('  - Are legitimate premium customers being flagged?')

print('\n\n3. SPECIFIC LIMITATIONS OF THIS ANALYSIS:')
print('-' * 100)
print('  Synthetic Data: This dataset was artificially generated')
print('  - Real customer behavior may be more complex')
print('  - Missing important features like tenure, age, location, device type')
print()
print('  Static Snapshot: This is a single point in time')
print('  - No information about trends or changes in behavior')
print('  - Cannot detect sudden shifts or seasonality')
print()
print('  Feature Engineering Assumptions: We created derived features')
print('  - Roaming ratio, data intensity may not capture all nuances')
print('  - Better features require domain expertise')

print('\n\n4. RECOMMENDATIONS FOR PRODUCTION USE:')
print('-' * 100)
print('  1. Combine with supervised learning: Use labels (churn, fraud) where available')
print('  2. Incorporate external data: Demographics, location, competitor actions')
print('  3. Monitor cluster drift: Re-cluster periodically and track segment changes')
print('  4. Human-in-the-loop: Always have domain experts review before acting')
print('  5. Gradual rollout: Test interventions on small groups first')
print('  6. Measure business impact: Track ROI of segment-based strategies')

print('\n\n' + '=' * 100)
print('CONCLUSION')
print('=' * 100)
print('This project demonstrated a complete unsupervised learning pipeline:')
print('  - Data preprocessing and feature engineering')
print('  - Multiple clustering algorithms (K-Means, Hierarchical, DBSCAN)')
print('  - Dimensionality reduction for visualization (PCA, UMAP)')
print('  - Anomaly detection with multiple methods')
print('  - Business translation and actionable insights')
print()
print('Key Takeaway: Unsupervised learning discovers patterns, but domain experts')
print('must validate and translate those patterns into business value.')
print('=' * 100)

MODEL LIMITATIONS AND VALIDATION REQUIREMENTS

1. INHERENT LIMITATIONS OF UNSUPERVISED LEARNING:
----------------------------------------------------------------------------------------------------
  No Ground Truth: Unlike supervised learning, we have no labels to validate against
  - Cannot calculate accuracy, precision, or recall
  - Must rely on internal metrics (silhouette score) and business validation

  Subjectivity in Interpretation: Mathematical clusters may not align with business intuition
  - Cluster 2 might split "premium users" based on subtle statistical differences
  - Domain experts must validate if distinctions are meaningful

  Parameter Sensitivity: Results depend heavily on algorithmic choices
  - K-Means requires choosing K (we used elbow + silhouette)
  - DBSCAN sensitive to eps and min_samples parameters
  - Different scalers or feature sets yield different clusters


2. VALIDATION REQUIREMENTS BEFORE DEPLOYMENT:
------------------------------------------------